In [1]:
import pandas as pd
df = pd.read_csv("car_listings_cleaned.csv")

df

,make,model,variant,car_price,body_type,miles,engine_vol,transmission,fuel_type,full_service,part_service,age
0,abarth,124 spider,gt,24275.0,convertible,10313.0,1.4,automatic,petrol,0,0,3.0
1,abarth,124 spider,gt,24275.0,convertible,10313.0,1.4,automatic,petrol,0,0,3.0
2,abarth,124 spider,gt,25000.0,convertible,11500.0,1.4,automatic,petrol,0,0,4.0
3,abarth,124 spider,multiair,15649.0,convertible,28692.0,1.4,manual,petrol,0,0,4.0
4,abarth,124 spider,multiair,15995.0,convertible,44000.0,1.4,manual,petrol,0,0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
749975,volvo,xc90,h t8,74900.0,suv,1072.0,2.0,automatic,petrol plug-in hybrid,0,0,0.0
749976,volvo,xc90,h t8,74900.0,suv,3091.0,2.0,automatic,petrol plug-in hybrid,0,0,0.0
749977,volvo,xc90,h t8,74995.0,suv,3077.0,2.0,automatic,petrol plug-in hybrid,0,0,0.0
749978,volvo,xc90,h t8,78900.0,suv,2015.0,2.0,automatic,petrol plug-in hybrid,0,0,0.0


In [4]:
df.corr(numeric_only=True)["car_price"]

car_price       1.000000
miles          -0.392616
engine_vol      0.496285
full_service   -0.035222
part_service   -0.045926
age            -0.401313
Name: car_price, dtype: float64

In [5]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder

ohe = OneHotEncoder(handle_unknown="ignore")
X_ohe = ohe.fit_transform(df[["make", "body_type", "transmission", "fuel_type"]])
target = TargetEncoder(cv=3, smooth="auto", random_state=42, target_type="continuous")
X = df[["model", "variant"]]
y = df["car_price"]
X_target = target.fit_transform(X, y)

X_ohe
X_target

array([[16003.12884523, 17855.11046819],
       [15770.41846602, 18080.21633407],
       [15855.3594063 , 19023.0068143 ],
       ...,
       [39429.22332571, 50943.33091069],
       [39691.07837424, 50977.98979809],
       [39429.22332571, 50943.33091069]])

In [10]:
import pandas as pd
from scipy.sparse import hstack

X_combined = hstack([
    X_ohe,
    X_target
])


ohe_names = ohe.get_feature_names_out([
    "make",
    "body_type",
    "transmission",
    "fuel_type"
])

target_names = [
    "model_target_encoded",
    "variant_target_encoded"
]

feature_names = list(ohe_names) + target_names

X_encoded_df = pd.DataFrame(
    X_combined.toarray(),
    columns=feature_names
)

# Add target
X_encoded_df["car_price"] = y.values

corr = (
    X_encoded_df.corr()["car_price"]
    .sort_values(ascending=False)
)

print(corr)

car_price                 1.000000
model_target_encoded      0.818087
variant_target_encoded    0.779186
transmission_automatic    0.377818
make_ferrari              0.293431
                            ...   
make_citroen             -0.069503
make_ford                -0.070505
make_vauxhall            -0.121312
body_type_hatchback      -0.251590
transmission_manual      -0.377818
Name: car_price, Length: 94, dtype: float64
